In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loading import cargar_csv

In [ ]:
df = pd.read_parquet("../mes1-python-datos/water_potability_limpio.parquet")
df.head()

Potability
0    1998
1    1278
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Potability"])
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")

Entrenamiento: 2620 muestras
Prueba: 656 muestras


In [5]:
print("Proporción en train:")
print(y_train.value_counts(normalize=True))

print("\nProporción en test:")
print(y_test.value_counts(normalize=True))

Proporción en train:
Potability
0    0.609924
1    0.390076
Name: proportion, dtype: float64

Proporción en test:
Potability
0    0.609756
1    0.390244
Name: proportion, dtype: float64


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Modelo entrenado")

Modelo entrenado


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.89      0.76       400
           1       0.63      0.30      0.41       256

    accuracy                           0.66       656
   macro avg       0.65      0.59      0.58       656
weighted avg       0.65      0.66      0.62       656



In [8]:
from sklearn.ensemble import IsolationForest

contamination_rate = y_train.value_counts(normalize=True)[1]

iso = IsolationForest(contamination=contamination_rate, random_state=42)
iso.fit(X_train)

iso_pred_raw = iso.predict(X_test)
y_pred_iso = np.where(iso_pred_raw == -1, 1, 0)

print(classification_report(y_test, y_pred_iso))

              precision    recall  f1-score   support

           0       0.62      0.66      0.64       400
           1       0.42      0.38      0.40       256

    accuracy                           0.55       656
   macro avg       0.52      0.52      0.52       656
weighted avg       0.54      0.55      0.55       656



In [9]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.66      0.93      0.77       400
           1       0.70      0.27      0.39       256

    accuracy                           0.67       656
   macro avg       0.68      0.60      0.58       656
weighted avg       0.68      0.67      0.62       656



In [10]:
from imblearn.over_sampling import SMOTE

print(f"Antes de SMOTE: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Después de SMOTE: {y_train_smote.value_counts().to_dict()}")

Antes de SMOTE: {0: 1598, 1: 1022}
Después de SMOTE: {0: 1598, 1: 1598}


In [11]:
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)

y_pred_rf_smote = rf_smote.predict(X_test)
print(classification_report(y_test, y_pred_rf_smote))

              precision    recall  f1-score   support

           0       0.70      0.75      0.72       400
           1       0.56      0.50      0.53       256

    accuracy                           0.65       656
   macro avg       0.63      0.62      0.63       656
weighted avg       0.65      0.65      0.65       656



In [12]:
smote_scaled = SMOTE(random_state=42)
X_train_scaled_smote, y_train_smote_svm = smote_scaled.fit_resample(X_train_scaled, y_train)

svm_smote = SVC(kernel="rbf", random_state=42)
svm_smote.fit(X_train_scaled_smote, y_train_smote_svm)

y_pred_svm_smote = svm_smote.predict(X_test_scaled)
print(classification_report(y_test, y_pred_svm_smote))

              precision    recall  f1-score   support

           0       0.69      0.67      0.68       400
           1       0.51      0.53      0.52       256

    accuracy                           0.62       656
   macro avg       0.60      0.60      0.60       656
weighted avg       0.62      0.62      0.62       656



In [14]:
import pandas as pd

resumen_parte_a = pd.DataFrame({
    "Modelo": ["Random Forest", "Random Forest + SMOTE", "Isolation Forest", "SVM", "SVM + SMOTE"],
    "Accuracy": [0.66, 0.65, 0.55, 0.67, 0.62],
    "Precision (clase 1)": [0.63, 0.56, 0.42, 0.70, 0.51],
    "Recall (clase 1)": [0.30, 0.50, 0.38, 0.27, 0.53],
    "F1 (clase 1)": [0.41, 0.53, 0.40, 0.39, 0.52],
})

resumen_parte_a

,Modelo,Accuracy,Precision (clase 1),Recall (clase 1),F1 (clase 1)
0,Random Forest,0.66,0.63,0.30,0.41
1,Random Forest + SMOTE,0.65,0.56,0.50,0.53
2,Isolation Forest,0.55,0.42,0.38,0.40
3,SVM,0.67,0.70,0.27,0.39
4,SVM + SMOTE,0.62,0.51,0.53,0.52


In [ ]:
df_temporal = pd.read_pickle("../mes1-python-datos/df_diario_backup.pkl") if False else None

fechas = pd.date_range(start="2024-01-01", end="2024-12-31", freq="h")
np.random.seed(42)
ph_valores = 7.2 + 0.5 * np.sin(np.linspace(0, 20, len(fechas))) + np.random.normal(0, 0.3, len(fechas))

df_sensor = pd.DataFrame({"timestamp": fechas, "ph": ph_valores}).set_index("timestamp")
df_temporal = df_sensor.resample("D").mean()

np.random.seed(7)
n_dias = len(df_temporal)
turbidez_valores = 3.5 + 1.5 * np.sin(np.linspace(0, 20, n_dias)) + np.random.normal(0, 0.8, n_dias)
df_temporal["turbidez"] = np.clip(turbidez_valores, 0.1, None)

df_temporal.head()

,ph,turbidez
timestamp,,
2024-01-01,7.168833,4.852421
2024-01-02,7.161589,3.209401
2024-01-03,7.296753,3.690311
2024-01-04,7.254957,4.071479
2024-01-05,7.336624,3.195003
